# MOLIERE v1.0

**Title:** A Method Of Lines Integrator for Emissions Research and Exploration (MOLIERE) v 1.0  
**LLNL release:** LLNL-CODE-2017385  
**CP number:** CP 2025-187  
**Author:** Steven A. Hawks  
**License:** MIT (`SPDX-License-Identifier: MIT`)

---
**v9 — non-uniform grid extension.** `run_simulation` now accepts optional boundary-refined
spatial grids via two new keyword arguments:

* `grid_scheme` — `'uniform'` (default), `'tanh'`, or `'geometric'`
* `grid_stretch` — stretch parameter (β for `'tanh'`, where the coarse-to-fine spacing
  ratio is cosh²β; the overall spacing ratio h_max/h_min for `'geometric'`)

The default (`'uniform'`) reproduces the previous (v8 speed-optimized) grid behavior. All
finite-difference stencils — including the analytical sparse Jacobian of the most common
case — are generated from three-point Lagrange differentiation formulas on the actual node
positions and reduce exactly to the former uniform-grid stencils when the grid is uniform.
A quantitative uniform-vs-refined comparison on the manuscript's Figure 1b benchmark is
included at the end of this notebook.

---
**v10 — conservative (Kirchhoff-flux) discretization.** The transport term is
integrated in conservative (divergence) form,
$x^{-m}\,\partial_x\!\left[x^{m} D\,\partial_x c\right]$, with a finite-volume
scheme built on the Kirchhoff potential $\psi(c) = \int D\,\mathrm{d}c = c_D\,D(c,T)$,
which is available in closed form for the exponential diffusivity law of Eqn. (7).
A chain-rule discretization of the equivalent form
$D\,\partial^2 c/\partial x^2 + (\mathrm{d}D/\mathrm{d}c)(\partial c/\partial x)^2$
is accurate for $c_D = \infty$ but becomes fragile when $c_D$ is small: $D$ then
varies by a factor of $e^{c_0/c_D}$ across the concentration front, the two terms
become large and nearly cancelling, and any overshoot feeds back exponentially
through $D \propto e^{c/c_D}$ — mass balance degrades and the stiff solver can fail
outright. The conservative scheme avoids this entirely.

The exact two-point face flux $-(\psi_{i+1}-\psi_i)/\Delta x = -c_D D_i\,\mathrm{expm1}(\Delta
c/c_D)/\Delta x$ requires no chain-rule expansion, interior fluxes telescope, and the
discrete solid + headspace inventory is conserved to ODE-solver tolerance for **any**
$c_D$, on uniform and non-uniform grids, for all geometries ($m = 0, 1, 2$) and both
boundary conditions (finite $F$ and $F = \infty$, including the $\mathrm{d}K/\mathrm{d}t$
repartitioning during temperature ramps). For $c_D = \infty$ it reduces to the standard
constant-$D$ flux; on a uniform grid the origin update reduces exactly to the
mirror-node formula. An analytical sparse Jacobian is supplied for the most common
case ($F = \infty$, $c_D = \infty$); the concentration-dependent cases use the
conservative sparsity pattern.

v10 also fixes a latent
integer-truncation bug in the initial-condition array (`np.full(N, c0free)` silently
created an integer array when `c0free` was an integer literal). A validation study —
mass conservation across geometries and a small-$c_D$ stress test — is appended at the
end of this notebook.

---
**v11 — signed heat-of-solution convention.** The partition-temperature parameter is now
`DeltaHs`, the signed thermodynamic enthalpy of sorption in kJ/mol, with negative values for
exothermic sorption. The notebook now evaluates
$K = K_0\exp[-\Delta H_s/(R_{\mathrm{gas}}T)]$ and
$(1/K)\,\mathrm{d}K/\mathrm{d}t = (\Delta H_s/R_{\mathrm{gas}}T^2)\,\mathrm{d}T/\mathrm{d}t$.
Thus the old positive-magnitude `EaK = +35` convention is replaced by
`DeltaHs = -35` for the same physical case. The transport term is discretized with the
conservative finite-volume (Kirchhoff-flux) scheme of v10.

---
**v12 — discretization cleanup.** The single-scheme cleanup removes two now-superseded
options so that the code path used in practice is the only one present:

* The `'sin'` spatial grid is removed; `grid_scheme` accepts `'uniform'` (default),
  `'tanh'`, and `'geometric'`.
* The legacy expanded (chain-rule) flux discretization and the `flux_scheme` keyword
  are removed. The conservative finite-volume Kirchhoff-flux scheme of v10 — which was
  already the default and is mass-conservative for any $c_D$ — is now the sole transport
  discretization. (Behavior of any prior `flux_scheme='conservative'` run is unchanged.)

All governing relations were re-verified: the signed-$\Delta H_s$ partition law and its
$\mathrm{d}K/\mathrm{d}t$ derivative, the exponential diffusivity law, the closed-form
Kirchhoff potential and `expm1` face flux, the finite-volume telescoping, and the
$F=\infty$ combined surface/headspace and finite-$F$ Robin boundary balances. Mass
conservation holds to better than $\sim\!10^{-3}\,\%$ across $m=0,1,2$, both boundary
conditions, and $c_D$ from $\infty$ down to small values.


In [ ]:
# SPDX-License-Identifier: MIT
# Copyright (c) 2026, Lawrence Livermore National Security, LLC
# LLNL-CODE-2017385
# CP 2025-187
# Author: Steven A. Hawks

import numpy as np
import matplotlib.pyplot as plt
import time


In [ ]:
from scipy.integrate import solve_ivp, simpson
from scipy.sparse import lil_matrix, csc_matrix, coo_matrix

# ============================================================================
# Vectorized helper functions (scalars AND arrays)
# ============================================================================

def temperature_vec(t, dt1, T0, T_target, RR):
    """Vectorized temperature profile -> (T_celsius, dTdt)."""
    t = np.asarray(t, dtype=float)
    scalar = t.ndim == 0
    t = np.atleast_1d(t)
    T = np.empty_like(t)
    dTdt = np.zeros_like(t)

    delta_T = T_target - T0
    if np.isclose(delta_T, 0.0):
        T.fill(T0)
    else:
        if RR == 0:
            raise ValueError('RR must be nonzero when T0 and Tfinal differ.')
        RR_eff = np.sign(delta_T) * abs(RR)
        t_ramp_end = dt1 + abs(delta_T) / abs(RR)
        m1 = t <= dt1
        m3 = t > t_ramp_end
        m2 = ~m1 & ~m3
        T[m1] = T0
        T[m2] = T0 + RR_eff * (t[m2] - dt1)
        dTdt[m2] = RR_eff
        T[m3] = T_target

    if scalar:
        return float(T[0]), float(dTdt[0])
    return T, dTdt


def flow_rate_vec(t, tEq, tFlush, Q_val):
    """Vectorized flow rate → Q array."""
    t = np.asarray(t, dtype=float)
    scalar = t.ndim == 0
    t = np.atleast_1d(t)
    Q = np.where(t % (tEq + tFlush) < tEq, 0.0, Q_val)
    return float(Q[0]) if scalar else Q


def feed_conc_vec(t, n_steps, delta, step_time, base_conc,
                  hold_time_initial, hold_time_final):
    """Vectorized feed concentration → ppbv array."""
    t = np.asarray(t, dtype=float)
    scalar = t.ndim == 0
    t = np.atleast_1d(t)
    total_time = hold_time_initial + 2 * n_steps * step_time + hold_time_final
    t_cycle = t % total_time
    result = np.full_like(t, base_conc)
    active = (t_cycle >= hold_time_initial) & (t_cycle < total_time - hold_time_final)
    if np.any(active):
        t_act = t_cycle[active] - hold_time_initial
        cs = (t_act / step_time).astype(int)
        rising = cs < n_steps
        result[active] = np.where(rising, base_conc + (cs + 1) * delta,
                                  base_conc + (2 * n_steps - cs - 1) * delta)
    return float(result[0]) if scalar else result




def _extract_temp_profile(params):
    """Return a normalized temperature profile tuple."""
    if 'Tfinal' in params:
        T_target = params['Tfinal']
    elif 'Tmax' in params:
        T_target = params['Tmax']
    else:
        raise KeyError("temp_params must define 'Tfinal' (preferred) or 'Tmax'.")
    return params['dt1'], params['T0'], T_target, params['RR']


def normalize_temp_params(params):
    """Return a copy of temp_params with both Tfinal and Tmax populated."""
    dt1, T0, T_target, RR = _extract_temp_profile(params)
    return {
        'dt1': dt1,
        'T0': T0,
        'Tfinal': T_target,
        'Tmax': T_target,
        'RR': RR,
    }


def validate_inputs(tfinal, temp_params, flow_params, cfeed_params,
                    m, R, mSample, rhoSample, Vvessel,
                    src_params, cD, F, cgas_init, N):
    """Validate user inputs and return a normalized src_params array."""
    src_params = np.asarray(src_params, dtype=float)

    if int(N) != N or N < 3:
        raise ValueError('N must be an integer >= 3.')
    if tfinal <= 0:
        raise ValueError('tfinal must be positive.')
    if R <= 0:
        raise ValueError('R must be positive.')
    if mSample <= 0 or rhoSample <= 0:
        raise ValueError('mSample and rhoSample must be positive.')
    if Vvessel <= mSample * 1e-3 / rhoSample:
        raise ValueError('Vvessel must be larger than the sample volume.')
    if src_params.size % 3 != 0:
        raise ValueError('src_params must contain groups of three values: c0, A, Ea.')
    if not np.isinf(cD) and cD <= 0:
        raise ValueError('cD must be positive or np.inf.')
    if not np.isinf(F) and F < 0:
        raise ValueError('F must be non-negative or np.inf.')
    if cgas_init < 0:
        raise ValueError('cgas_init must be non-negative.')
    if m not in (0, 1, 2):
        raise ValueError('m must be 0 (slab), 1 (cylinder), or 2 (sphere).')

    dt1, T0, T_target, RR = _extract_temp_profile(temp_params)
    if RR == 0 and not np.isclose(T_target, T0):
        raise ValueError('RR must be nonzero when T0 and Tfinal differ.')
    if flow_params['tEq'] + flow_params['tFlush'] <= 0:
        raise ValueError('tEq + tFlush must be positive.')

    n_steps = cfeed_params['n_steps']
    if int(n_steps) != n_steps or n_steps < 0:
        raise ValueError('n_steps must be a non-negative integer.')
    if n_steps > 0 and cfeed_params['step_time'] <= 0:
        raise ValueError('step_time must be positive when n_steps > 0.')
    total_feed_time = (
        cfeed_params['hold_time_initial']
        + 2 * int(n_steps) * cfeed_params['step_time']
        + cfeed_params['hold_time_final']
    )
    if total_feed_time <= 0:
        raise ValueError('The total feed profile duration must be positive.')

    return src_params


# ============================================================================
# Sparse Jacobian sparsity pattern
# ============================================================================

def generate_jacobian_sparsity(N, num_srcs, is_inf_F):
    """Build Jacobian sparsity as sparse CSC matrix."""
    total = N + num_srcs if is_inf_F else N + 1 + num_srcs
    sp = lil_matrix((total, total), dtype=np.int8)
    for i in range(N):
        sp[i, i] = 1
    for i in range(1, N - 1):
        sp[i, i - 1] = 1
        sp[i, i + 1] = 1
    if N > 1:
        sp[0, 1] = 1
    if is_inf_F:
        if N - 1 >= 1: sp[N - 1, N - 2] = 1
        if N - 1 >= 2: sp[N - 1, N - 3] = 1
        for si in range(num_srcs):
            sp[N + si, N + si] = 1
            sp[:N - 1, N + si] = 1
    else:
        if N - 1 >= 1: sp[N - 1, N - 2] = 1
        sp[N - 1, N] = 1
        sp[N, N - 1] = 1
        sp[N, N] = 1
        for si in range(num_srcs):
            row = N + 1 + si
            sp[row, row] = 1
            sp[:N, row] = 1
    return csc_matrix(sp)


# ============================================================================
# Spatial grid generation (uniform or boundary-refined non-uniform)
# ============================================================================

def make_grid(R, N, scheme='uniform', stretch=2.0):
    """
    Generate the spatial grid x[0..N-1] on [0, R].

    For outgassing/desorption problems the concentration gradient is steepest
    at the sample surface (x = R) at early times, while the profile is flat at
    the symmetry point (x = 0).  A boundary-refined grid therefore places
    nodes densely near x = R and sparsely near x = 0.

    Schemes
    -------
    'uniform'    : x_i = R * i/(N-1).  (Default; reproduces previous behavior.)
    'tanh'       : Roberts-type one-sided stretching,
                   x = R * tanh(stretch * xi) / tanh(stretch),  xi in [0, 1].
                   `stretch` = beta > 0 controls clustering at x = R; the
                   coarse-to-fine spacing ratio is h_max/h_min = cosh^2(beta)
                   (beta = 1.5 -> ~5.5x, beta = 2 -> ~14x, beta = 2.5 -> ~38x).
                   Smooth (analytic) mapping, so adjacent spacings vary
                   gradually and the 3-point stencils retain ~2nd-order
                   behavior.  Recommended non-uniform scheme.
    'geometric'  : spacings form a geometric progression, refined toward
                   x = R; `stretch` = h_max/h_min (> 1) is the total
                   coarse-to-fine spacing ratio.

    Returns
    -------
    ndarray of shape (N,), strictly increasing, with x[0] = 0 and x[-1] = R.
    """
    xi = np.linspace(0.0, 1.0, N)
    if scheme == 'uniform':
        x = R * xi
    elif scheme == 'tanh':
        beta = float(stretch)
        if beta <= 0:
            raise ValueError("stretch (beta) must be > 0 for scheme='tanh'.")
        x = R * np.tanh(beta * xi) / np.tanh(beta)
    elif scheme == 'geometric':
        s = float(stretch)
        if s <= 0:
            raise ValueError("stretch must be > 0 for scheme='geometric'.")
        if np.isclose(s, 1.0):
            x = R * xi
        else:
            q = s ** (-1.0 / (N - 2))           # per-interval ratio (< 1 refines toward R)
            hs = q ** np.arange(N - 1)
            hs *= R / hs.sum()
            x = np.concatenate(([0.0], np.cumsum(hs)))
    else:
        raise ValueError("scheme must be 'uniform', 'tanh', or 'geometric'.")
    x[0] = 0.0
    x[-1] = R          # enforce exact endpoints against floating-point drift
    if np.any(np.diff(x) <= 0):
        raise ValueError('Generated grid is not strictly increasing.')
    return x


def nonuniform_fd_coeffs(x, m_geom):
    """
    Precompute 3-point finite-difference coefficient arrays on the (possibly
    non-uniform) grid x.  On a uniform grid every formula below reduces
    exactly to the familiar central/one-sided second-order stencils.

    Interior nodes i = 1..N-2, with h- = x_i - x_{i-1} and h+ = x_{i+1} - x_i:
        dc/dx|_i     ~ a1*c_{i-1} + b1*c_i + c1*c_{i+1}      (2nd order)
        d2c/dx2|_i   ~ a2*c_{i-1} + b2*c_i + c2*c_{i+1}      (2nd order on
                       smoothly varying grids; the leading truncation term is
                       proportional to (h+ - h-), which is O(h^2) for the
                       smooth mappings produced by make_grid)
    The constant-D transport operator D*(d2c/dx2 + (m/x) dc/dx) is also
    returned pre-combined as the tridiagonal arrays (Lsub, Ldia, Lsup).

    Origin (x = 0): the symmetry condition c'(0) = 0 with a mirror ghost node
    at -h1 gives d2c/dx2|_0 ~ 2*(c_1 - c_0)/h1^2 (2nd order; odd derivatives
    vanish at the symmetry point), and L'Hopital gives
    (m/x)*dc/dx -> m*d2c/dx2, so dc/dt|_0 = (m+1)*D*2*(c_1 - c_0)/h1^2.

    Surface (x = R): one-sided 3-point (2nd-order) derivative
        dc/dx|_R ~ wA*c_{N-3} + wB*c_{N-2} + wC*c_{N-1},
    which reduces to (3c_{N-1} - 4c_{N-2} + c_{N-3})/(2h) on a uniform grid.
    """
    hm = x[1:-1] - x[:-2]
    hp = x[2:] - x[1:-1]
    s = hm + hp
    a1 = -hp / (hm * s)
    b1 = (hp - hm) / (hm * hp)
    c1 = hm / (hp * s)
    a2 = 2.0 / (hm * s)
    b2 = -2.0 / (hm * hp)
    c2 = 2.0 / (hp * s)
    x_int = x[1:-1]
    g = m_geom / x_int if m_geom != 0 else np.zeros_like(x_int)
    Lsub = a2 + g * a1
    Ldia = b2 + g * b1
    Lsup = c2 + g * c1
    h1 = x[1] - x[0]                     # origin spacing
    e1 = x[-1] - x[-2]                   # last spacing (at the surface)
    e2 = x[-2] - x[-3]
    wA = e1 / (e2 * (e1 + e2))
    wB = -(e1 + e2) / (e1 * e2)
    wC = (2.0 * e1 + e2) / (e1 * (e1 + e2))
    return dict(a1=a1, b1=b1, c1=c1, a2=a2, b2=b2, c2=c2,
                Lsub=Lsub, Ldia=Ldia, Lsup=Lsup,
                g=g, h1=h1, hb=e1, wA=wA, wB=wB, wC=wC)


# ============================================================================
# Conservative (Kirchhoff-flux) finite-volume discretization        [v10]
# ============================================================================
#
# The transport term is discretized in conservative (divergence) form rather
# than the chain-rule form  dc/dt = dD/dc (dc/dx)^2 + D (d2c/dx2 + (m/x) dc/dx).
# With D = D0 exp(c/cD - Ea,D/RgasT) and small cD, D varies by orders of
# magnitude across a few nodes; a chain-rule discretization is then the
# difference of two large, nearly cancelling terms, can overshoot (c > c0
# makes D explode as e^{c/cD}) or oscillate to failure, and does not conserve
# mass.  The conservative scheme below avoids all of this by construction.
#
# It discretizes the divergence form directly,
#
#     dc/dt = (1/x^m) d/dx [ x^m * (-q) ],     q = -D dc/dx = -dpsi/dx,
#
# using the Kirchhoff potential  psi(c) = INT_0^c D(c',T) dc'.  For the
# exponential law used here psi has the closed form
#
#     psi(c) = cD * D(c,T) + const   =>   psi_{i+1} - psi_i = cD (D_{i+1} - D_i)
#                                                           = cD D_i expm1(dc/cD),
#
# so the exact two-point flux between adjacent nodes is available in closed
# form with no chain-rule expansion (the expm1 form is used for numerical
# stability when |dc|/cD is small).  As cD -> inf this reduces to the usual
# constant-D flux  -D (c_{i+1}-c_i)/dx.  Each node i owns the control volume
# between the midpoint faces x_{i-1/2}, x_{i+1/2} (half cells at x = 0 and
# x = R), with face areas x_f^m and volumes V_i = (x_r^{m+1}-x_l^{m+1})/(m+1).
# Interior fluxes telescope, so the discrete solid inventory  sum_i V_i c_i
# changes only through the surface flux and the source terms: mass is
# conserved to ODE-solver tolerance for ANY cD, on uniform and non-uniform
# grids alike.  On a uniform grid with constant D the origin update reduces
# exactly to the mirror-node formula dc/dt|_0 = 2(m+1)D(c1-c0)/h^2.
#
# Boundary treatment:
#   * F = inf : the surface cell and the headspace form a single equilibrium
#     reservoir (c_gas = c_R / K).  Their combined mass balance gives
#         [gam*V_{N-1} + Vhs/K] dc_R/dt = gam*Fl_{N-3/2}
#                                         - Q (c_R/K - c_feed)
#                                         + c_R (Vhs/K) (1/K)(dK/dt)
#                                         + gam*V_{N-1} * src_total,
#     which reduces to the well-mixed surface limit (Eqn 12) as the surface
#     cell shrinks, and conserves (solid + headspace) mass exactly, including
#     the dK/dt repartitioning during temperature ramps.
#   * finite F : the Robin flux  q_R = F (c_R - K c_gas)  is applied at the
#     outer face of the surface cell, and the identical flux feeds the
#     headspace ODE, Eqn (5) -- exact solid/gas mass exchange by construction.


def fv_geometry(x, m_geom):
    """Finite-volume geometry on the (possibly non-uniform) grid x.

    Node i owns the control volume between the midpoint faces; the first and
    last cells are half cells touching x = 0 and x = R.

    Returns
    -------
    xf : ndarray, shape (N-1,) -- interior face positions (midpoints)
    Af : ndarray, shape (N-1,) -- face area factors x_f^m
    V  : ndarray, shape (N,)   -- cell volumes INT x^m dx over each cell;
                                  sum(V) = R^(m+1)/(m+1) exactly (telescoping)
    """
    xf = 0.5 * (x[:-1] + x[1:])
    xl = np.concatenate(([x[0]], xf))
    xr = np.concatenate((xf, [x[-1]]))
    mp1 = m_geom + 1
    V = (xr ** mp1 - xl ** mp1) / mp1
    Af = xf ** m_geom if m_geom != 0 else np.ones_like(xf)
    return xf, Af, V


def generate_jacobian_sparsity_cons(N, num_srcs, is_inf_F):
    """Jacobian sparsity for the conservative (Kirchhoff-flux) scheme.

    Relative to the F = inf finite-difference pattern this differs in two ways:
    the surface row couples only to (N-2, N-1) -- the two-point face flux
    replaces the one-sided 3-point gradient -- and the source columns also feed
    the surface row (generation inside the surface cell is bookkept).
    """
    if not is_inf_F:
        return generate_jacobian_sparsity(N, num_srcs, False)
    total = N + num_srcs
    sp = lil_matrix((total, total), dtype=np.int8)
    for i in range(N):
        sp[i, i] = 1
    for i in range(1, N - 1):
        sp[i, i - 1] = 1
        sp[i, i + 1] = 1
    if N > 1:
        sp[0, 1] = 1
        sp[N - 1, N - 2] = 1
    for si in range(num_srcs):
        sp[N + si, N + si] = 1
        sp[:N, N + si] = 1
    return csc_matrix(sp)


def make_ode_system_conservative(N, x, R, cD, beta, dt1, T0, Tmax, RR, tEq,
                                 tFlush, Q_val, n_steps, delta, step_time,
                                 base_conc, hold_time_initial, hold_time_final,
                                 src_params, Vheadspace, F, Rgas, D0, EaD, K0,
                                 DeltaHs, m_geom):
    """Build the conservative (Kirchhoff-flux) finite-volume ODE system.

    Returns (rhs_func, solver_kwargs).
    An analytical sparse Jacobian is supplied for the most common case
    (F = inf, cD = inf); concentration-dependent cases fall back to the
    numerical Jacobian with the conservative sparsity pattern.
    """
    num_srcs = len(src_params) // 3
    is_inf_F = np.isinf(F)
    is_inf_cD = np.isinf(cD)

    # ---- Finite-volume geometry (uniform or non-uniform grid) ----
    mp1 = m_geom + 1
    xf, Af, V = fv_geometry(x, m_geom)
    dx = np.diff(x)
    Af_dx = Af / dx                       # face area / node spacing, (N-1,)
    invV = 1.0 / V
    vSample = Vheadspace / beta
    gam = vSample * mp1 / R ** mp1        # ml per unit of INT c x^m dx
    gamV_last = gam * V[-1]               # physical volume of surface cell (ml)

    # Conservative constant-D operator, pre-combined (tridiagonal):
    #   dc_i/dt = D * (Csub*c_{i-1} + Cdia*c_i + Csup*c_{i+1}),  i = 1..N-2
    Csub = Af_dx[:-1] * invV[1:N - 1]
    Csup = Af_dx[1:] * invV[1:N - 1]
    Cdia = -(Csub + Csup)
    orgC = Af_dx[0] * invV[0]             # origin: dc/dt|_0 = orgC * D * (c1 - c0)

    inv_R_beta = 1.0 / (R * beta)
    inv_Vhs = 1.0 / Vheadspace
    neg_EaD_Rgas = -EaD / Rgas
    DeltaHs_Rgas = DeltaHs / Rgas
    delta_T = Tmax - T0
    if np.isclose(delta_T, 0.0):
        RR_eff = 0.0
        t_ramp_end = dt1
    else:
        RR_eff = np.sign(delta_T) * abs(RR)
        t_ramp_end = dt1 + abs(delta_T) / abs(RR)
    period = tEq + tFlush
    total_feed_time = hold_time_initial + 2 * n_steps * step_time + hold_time_final
    inv_cD = 0.0 if is_inf_cD else 1.0 / cD

    if num_srcs > 0:
        src_A = src_params[1::3].copy()
        src_Ea = src_params[2::3].copy()
    else:
        src_A = np.empty(0)
        src_Ea = np.empty(0)

    n_total = (N + num_srcs) if is_inf_F else (N + 1 + num_srcs)

    # ---- Inline scalar helpers ----
    def _temp(t):
        if t <= dt1:
            return T0, 0.0
        if t <= t_ramp_end:
            return T0 + RR_eff * (t - dt1), RR_eff
        return Tmax, 0.0

    def _flow(t):
        return 0.0 if (t % period) < tEq else Q_val

    def _feed(t):
        if n_steps == 0: return base_conc
        tc = t % total_feed_time
        if tc < hold_time_initial or tc >= total_feed_time - hold_time_final:
            return base_conc
        cs = int((tc - hold_time_initial) / step_time)
        return base_conc + (cs + 1) * delta if cs < n_steps else base_conc + (2 * n_steps - cs - 1) * delta

    def _dpsi(cs, inv_T):
        """Kirchhoff potential differences psi_{i+1} - psi_i at all faces."""
        if is_inf_cD:
            D_T = D0 * np.exp(neg_EaD_Rgas * inv_T)
            return D_T * np.diff(cs)
        Dn = D0 * np.exp(cs * inv_cD + neg_EaD_Rgas * inv_T)
        return cD * Dn[:-1] * np.expm1(np.diff(cs) * inv_cD)

    # ==================================================================
    # CASE 1: isinf(F)  --  equilibrium surface (combined cell/headspace)
    # ==================================================================
    if is_inf_F:
        src_off = N

        def rhs(t, c):
            TC, dTdt_val = _temp(t)
            T = TC + 273.15
            inv_T = 1.0 / T
            Q = _flow(t)
            cfeed = _feed(t) * 0.001 / 22.414 * 273.15 * inv_T

            K = K0 * np.exp(-DeltaHs_Rgas * inv_T)
            dlnK_dt = DeltaHs_Rgas * dTdt_val * inv_T * inv_T

            cs = c[:N]
            Fl = -Af_dx * _dpsi(cs, inv_T)   # area-weighted face fluxes (+x)

            src_total = 0.0
            if num_srcs > 0:
                rates = src_A * np.exp(-src_Ea / (Rgas * T)) * c[src_off:src_off + num_srcs]
                src_total = float(np.sum(rates))

            dcdt = np.empty(n_total)
            dcdt[0] = -Fl[0] * invV[0] + src_total
            dcdt[1:N - 1] = (Fl[:-1] - Fl[1:]) * invV[1:N - 1] + src_total

            # Combined surface-cell + headspace balance (c_gas = c_R / K)
            VhK = Vheadspace / K
            dcdt[N - 1] = (gam * Fl[-1]
                           - Q * (cs[-1] / K - cfeed)
                           + cs[-1] * VhK * dlnK_dt
                           + gamV_last * src_total) / (gamV_last + VhK)

            if num_srcs > 0:
                dcdt[src_off:src_off + num_srcs] = -rates
            return dcdt

        if not is_inf_cD:
            sparsity = generate_jacobian_sparsity_cons(N, num_srcs, True)
            return rhs, dict(jac_sparsity=sparsity)

        # ---- Analytical Jacobian for the constant-D case ----
        idx_int = np.arange(1, N - 1)
        jac_rows = np.concatenate([
            [0, 0],                                          # center
            idx_int, idx_int, idx_int,                        # interior sub/diag/super
            [N - 1, N - 1],                                  # surface (2-pt flux)
            np.arange(src_off, src_off + num_srcs),          # source self
            np.tile(np.arange(N), num_srcs),                 # source -> all solid rows
        ]).astype(np.int32)
        jac_cols = np.concatenate([
            [0, 1],
            idx_int - 1, idx_int, idx_int + 1,
            [N - 2, N - 1],
            np.arange(src_off, src_off + num_srcs),
            np.repeat(np.arange(src_off, src_off + num_srcs), N),
        ]).astype(np.int32)
        n_entries = len(jac_rows)
        _jac_vals = np.empty(n_entries)

        def jac(t, c):
            TC, dTdt_val = _temp(t)
            T = TC + 273.15
            inv_T = 1.0 / T
            Q = _flow(t)

            K = K0 * np.exp(-DeltaHs_Rgas * inv_T)
            dlnK_dt = DeltaHs_Rgas * dTdt_val * inv_T * inv_T
            D = D0 * np.exp(neg_EaD_Rgas * inv_T)
            VhK = Vheadspace / K
            inv_denomS = 1.0 / (gamV_last + VhK)

            vals = _jac_vals
            v = orgC * D
            vals[0] = -v
            vals[1] = v

            base = 2
            n_int = N - 2
            vals[base:base + n_int] = D * Csub
            vals[base + n_int:base + 2 * n_int] = D * Cdia
            vals[base + 2 * n_int:base + 3 * n_int] = D * Csup

            base_bnd = base + 3 * n_int
            a = gam * Af_dx[-1] * D * inv_denomS
            vals[base_bnd] = a                                              # d/dc_{N-2}
            vals[base_bnd + 1] = -a + (-Q / K + VhK * dlnK_dt) * inv_denomS  # d/dc_{N-1}

            base_src = base_bnd + 2
            if num_srcs > 0:
                k_src = src_A * np.exp(-src_Ea / (Rgas * T))
                vals[base_src:base_src + num_srcs] = -k_src
                base_coup = base_src + num_srcs
                w_last = gamV_last * inv_denomS
                for si in range(num_srcs):
                    blk = vals[base_coup + si * N:base_coup + (si + 1) * N]
                    blk[:N - 1] = k_src[si]
                    blk[N - 1] = w_last * k_src[si]

            return csc_matrix(coo_matrix(
                (vals.copy(), (jac_rows, jac_cols)), shape=(n_total, n_total)))

        return rhs, dict(jac=jac)

    # ==================================================================
    # CASE 2: finite F  --  Robin flux applied at the outer face
    # ==================================================================
    else:
        src_off = N + 1
        Rm = R ** m_geom

        def rhs(t, c):
            TC, _ = _temp(t)
            T = TC + 273.15
            inv_T = 1.0 / T
            Q = _flow(t)
            cfeed = _feed(t) * 0.001 / 22.414 * 273.15 * inv_T

            K = K0 * np.exp(-DeltaHs_Rgas * inv_T)
            cs = c[:N]
            cR = cs[-1]
            c_gas = c[N]

            Fl = -Af_dx * _dpsi(cs, inv_T)
            qR = 60.0 * F * (cR - c_gas * K)     # outward surface flux (Eqn 4)

            src_total = 0.0
            if num_srcs > 0:
                rates = src_A * np.exp(-src_Ea / (Rgas * T)) * c[src_off:src_off + num_srcs]
                src_total = float(np.sum(rates))

            dcdt = np.empty(n_total)
            dcdt[0] = -Fl[0] * invV[0] + src_total
            dcdt[1:N - 1] = (Fl[:-1] - Fl[1:]) * invV[1:N - 1] + src_total
            dcdt[N - 1] = (Fl[-1] - Rm * qR) * invV[N - 1] + src_total
            dcdt[N] = (mp1 * 60.0 * F * (cR - c_gas * K) * inv_R_beta
                       + Q * inv_Vhs * (cfeed - c_gas))

            if num_srcs > 0:
                dcdt[src_off:src_off + num_srcs] = -rates
            return dcdt

        sparsity = generate_jacobian_sparsity_cons(N, num_srcs, False)
        return rhs, dict(jac_sparsity=sparsity)


# ============================================================================
# Main simulation
# ============================================================================

def run_simulation(tfinal, temp_params, flow_params, cfeed_params,
                   m, R, mSample, rhoSample, Vvessel, MW_analyte,
                   src_params, DeltaHs, K50, D50, EaD, cD, F, c0free, cgas_init, N,
                   grid_scheme='uniform', grid_stretch=2.0, verbose=True):
    """Run diffusion simulation with the given parameters.

    The transport term is integrated with the conservative finite-volume
    Kirchhoff-flux discretization of (1/x^m) d/dx(x^m D dc/dx): mass-conservative
    by construction (to ODE-solver tolerance) for any cD, all geometries
    (m = 0, 1, 2), and both boundary conditions (finite F and F = inf).

    grid_scheme : 'uniform' (default), 'tanh', or 'geometric'
        Spatial grid; non-uniform schemes refine toward the surface (x = R).
        See make_grid for details.
    grid_stretch : float
        Stretch parameter (beta for 'tanh'; h_max/h_min for 'geometric').
    verbose : bool
        Print solve time, grid info, and mass-balance error.
    """
    src_params = validate_inputs(
        tfinal, temp_params, flow_params, cfeed_params,
        m, R, mSample, rhoSample, Vvessel,
        src_params, cD, F, cgas_init, N,
    )
    temp_params = normalize_temp_params(temp_params)
    num_srcs = len(src_params) // 3
    is_inf_F = np.isinf(F)
    vSample = mSample * 1E-3 / rhoSample
    Vheadspace = Vvessel - vSample
    beta = Vheadspace / vSample
    Rgas = 8.31446261815324 / 1000
    K0 = K50 * np.exp(DeltaHs / Rgas / (50 + 273.15))
    D0 = 60 * D50 * np.exp(EaD / Rgas / (50 + 273.15))

    x = make_grid(R, N, scheme=grid_scheme, stretch=grid_stretch)
    if verbose and grid_scheme != 'uniform':
        hs = np.diff(x)
        print(f"Grid: '{grid_scheme}' (stretch = {grid_stretch:g}), "
              f"h_min = {hs.min():.3e} cm at x = R, h_max = {hs.max():.3e} cm, "
              f"ratio = {hs.max() / hs.min():.1f}")
    K_t0 = K0 * np.exp(-DeltaHs / Rgas / (temp_params['T0'] + 273.15))

    # Conservative finite-volume geometry (the only transport discretization).
    _, _, Vfv = fv_geometry(x, m)
    gam = vSample * (m + 1) / R ** (m + 1)

    # Initial conditions (dtype=float guards against silent integer
    # truncation of the boundary value when c0free is passed as an int)
    c_init = np.full(N, c0free, dtype=float)
    if is_inf_F:
        # Mass-consistent IC for the combined surface-cell/headspace state:
        # the surface cell equilibrates with the headspace at t = 0+, so the
        # combined inventory gam*V[-1]*c0free + Vheadspace*cgas_init is
        # preserved exactly.  Reduces to cgas_init*K_t0 as the cell shrinks,
        # and to c0free when the system starts pre-equilibrated.
        c_init[-1] = ((gam * Vfv[-1] * c0free + Vheadspace * cgas_init)
                      / (gam * Vfv[-1] + Vheadspace / K_t0))
        ic = np.hstack((c_init, src_params[::3])) if num_srcs > 0 else c_init
    else:
        ic = np.hstack((c_init, [cgas_init], src_params[::3])) if num_srcs > 0 else np.hstack((c_init, [cgas_init]))

    # Extract params once
    dt1, T0, Tmax, RR = _extract_temp_profile(temp_params)
    tEq, tFlush, Q_val = flow_params['tEq'], flow_params['tFlush'], flow_params['Q']
    ns = cfeed_params['n_steps']
    dl = cfeed_params['delta']
    st = cfeed_params['step_time']
    bc = cfeed_params['base_conc']
    hti = cfeed_params['hold_time_initial']
    htf = cfeed_params['hold_time_final']

    # Build specialized ODE system (conservative finite-volume discretization)
    rhs, solver_kw = make_ode_system_conservative(
        N, x, R, cD, beta, dt1, T0, Tmax, RR, tEq, tFlush, Q_val,
        ns, dl, st, bc, hti, htf, src_params, Vheadspace, F, Rgas, D0, EaD, K0, DeltaHs, m)

    # Solve
    t0 = time.time()
    result = solve_ivp(rhs, [0, tfinal], ic, method='BDF',
                       rtol=1e-7, atol=1e-8, max_step=tfinal / 10, **solver_kw)
    elapsed = time.time() - t0
    if verbose:
        print(f"Solve time: {elapsed:.4f} seconds")

    if not result.success:
        raise RuntimeError(f'solve_ivp failed: {result.message}')

    t = result.t
    y = result.y.T
    cR = y[:, N - 1]

    # ---- Vectorized post-processing ----
    T_C, _ = temperature_vec(t, dt1, T0, Tmax, RR)
    Q = flow_rate_vec(t, tEq, tFlush, Q_val)
    T_K = T_C + 273.15
    K = K0 * np.exp(-DeltaHs / Rgas / T_K)
    S = K * (273.15 / T_K)
    DR = D0 * np.exp(-EaD / Rgas / T_K + cR * (0.0 if np.isinf(cD) else 1.0 / cD))

    c_feed_ppbv = feed_conc_vec(t, ns, dl, st, bc, hti, htf)
    c_feed_uM = c_feed_ppbv / 1000 / 22.414 * 273.15 / T_K
    c_feed_ugm3 = c_feed_uM * 1000 * MW_analyte

    c_gas = cR / K if is_inf_F else y[:, N]
    cGas_ugm3 = c_gas * 1000 * MW_analyte
    cGasPPBV = c_gas * 1000 * 22.414 / 273.15 * T_K
    cGasPPBVEq = c0free / (K + beta) * 1000 * 22.414 / 273.15 * T_K

    # Solid-phase inventory integral INT c x^m dx, evaluated with the FV cell
    # volumes that are the conservative scheme's exact discrete invariant.
    initial_integral = y[0, :N] @ Vfv
    time_integrals = y[:, :N] @ Vfv
    delta_mass_free = -(m + 1) / R ** (m + 1) * vSample * MW_analyte * (initial_integral - time_integrals)

    src_off = N if is_inf_F else N + 1
    if num_srcs > 0:
        delta_mass_src = -vSample * MW_analyte * sum(
            src_params[3 * j] - y[:, src_off + j] for j in range(num_srcs))
    else:
        delta_mass_src = np.zeros_like(t)

    delta_mass_total = delta_mass_free + delta_mass_src

    # Mass balance check
    c0_end = (m + 1) / R ** (m + 1) * (y[-1, :N] @ Vfv)
    c_Gas_end = c_gas[-1] * beta
    initial_total = c0free + beta * cgas_init + float(np.sum(src_params[::3]))
    final_source_total = float(np.sum(y[-1, src_off:src_off + num_srcs])) if num_srcs > 0 else 0.0
    final_total = c0_end + c_Gas_end + final_source_total
    denom = initial_total - final_total

    if np.abs(denom) < 1E-3:
        c0Chk = simpson(Q * (c_gas - c_feed_uM) / vSample, x=t)
        mb_error_pct = 100 * c0Chk
        if verbose:
            print("Warning: denominator nearly zero. Using absolute error metric instead.")
            print(f'Mass Balance Error = {mb_error_pct:.2g} %')
    else:
        c0Chk = simpson(Q * (c_gas - c_feed_uM) / vSample, x=t) / denom
        mb_error_pct = 100 * (1 - c0Chk)
        if verbose:
            print(f'Mass Balance Error = {mb_error_pct:.2g} %')

    return {
        't (min)': t,
        'c_gas (uM)': c_gas,
        'c_gas (ug/m^3)': cGas_ugm3,
        'c_gas (ppbv)': cGasPPBV,
        'T (C)': T_C,
        'Δm (ng)': delta_mass_total,
        'cGasEq (ppbv)': cGasPPBVEq,
        'Q (ml/min)': Q,
        'c_feed (uM)': c_feed_uM,
        'c_feed (ppbv)': c_feed_ppbv,
        'c_feed (ug/m^3)': c_feed_ugm3,
        'K': K,
        'S (cm^3 @ STP/atm/cm^3 sample)': S,
        'D @ R (cm^2/min)': DR,
        'x (cm)': x,
        'solve_time (s)': elapsed,
        'mass_balance_error (%)': mb_error_pct,
    }


In [ ]:
# SIMULATION PARAMETERS
tfinal = 150  # Final simulation time (min)
N = 201 # number of grid points

# Temperature Profile parameters
temp_params = {
    'dt1': 60,    # min (initial isothermal period)
    'T0': 50,       # °C, initial temperature
    'Tfinal': 250,    # °C, final temperature
    'RR': 5      # °C/min, ramp rate
}

# Flow parameters
flow_params = {
    'tEq': 30,         # min, time at which Q = 0
    'tFlush': tfinal, # min, flushing time (when Q becomes active)
    'Q': 20           # ml/min, flow rate
}

# Feed concentration profile parameters
cfeed_params = {
    'n_steps': 0,               # number of steps
    'delta': 1000,              # ppbv, concentration change per step
    'step_time': 240,           # min, time for each step
    'base_conc': 0,             # ppbv, base concentration
    'hold_time_initial': 60,    # min, initial hold time
    'hold_time_final': 3000     # min, final hold time
}

# Sample and vessel characteristics
m = 2               # 0 = slab, 1 = cylinder, 2 = sphere
R = 200e-4          # cm, sample radius
mSample = 50        # mg, mass of sample
rhoSample = 1       # g/ml, density of sample
Vvessel = 10        # ml, vessel volume
MW_analyte = 18.0   # g/mol, molecular weight of analyte

# Source parameters: empty array for no chemical sources
# src_params = np.array([]) # Empty = no sources
# Example with sources:
src_params = np.array([100, 1E8, 80])  # list of: c0 (uM), A (1/min), Ea (kJ/mol), c0 (uM),...etc.

# Other parameters
DeltaHs = -35     # kJ/mol, signed enthalpy of sorption (negative for exothermic sorption)
K50 = 150     # K at 50 °C
D50 = 1E-7    # cm^2/s at 50 °C
EaD = 15      # kJ/mol
cD = np.inf      # uM, plasticizer power parameter
F = np.inf      # cm/s; use np.inf for infinite flow condition
c0free = 10  # uM, initial free concentration
cgas_init = 0 # uM, initial gas concentration

# Spatial grid options (v9): non-uniform schemes refine toward the surface (x = R)
grid_scheme = 'uniform'   # 'uniform' (default) | 'tanh' | 'geometric'
grid_stretch = 2.0        # 'tanh': beta (spacing ratio = cosh^2(beta)); 'geometric': h_max/h_min


In [ ]:
# Run the simulation
results = run_simulation(tfinal, temp_params, flow_params, cfeed_params,
                         m, R, mSample, rhoSample, Vvessel, MW_analyte,
                         src_params, DeltaHs, K50, D50, EaD, cD, F, c0free, cgas_init, N,
                         grid_scheme=grid_scheme, grid_stretch=grid_stretch)

# Plot gas concentration
plt.figure(figsize=(10, 6))
plt.plot(results['t (min)'], results['c_gas (ppbv)'], label='c_gas', linewidth=3)
plt.xlim([np.min(results['t (min)']), np.max(results['t (min)'])])
plt.xlabel('Time (min)', fontsize=18)
plt.ylabel('Gas Concentration (ppbv)', fontsize=18)
plt.tick_params(axis='both', which='major', top=True, right=True, labelsize=16,
                direction='in', length=6, width=2)
plt.tick_params(axis='y', which='minor', right=True, direction='in', length=4, width=1.5)
plt.xlim(left=-1)
plt.ylim(bottom=0)
ax = plt.gca()
ax.spines['top'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
plt.tight_layout()
plt.show()

# Plot sample mass change
plt.figure(figsize=(10, 6))
plt.plot(results['t (min)'], results['Δm (ng)'], 
         label='Sample mass change', linewidth=3, color='magenta')
plt.xlabel('Time (min)', fontsize=16)
plt.ylabel('$\\Delta$ Sample Mass (ng)', fontsize=16)
plt.xlim([np.min(results['t (min)']), np.max(results['t (min)'])])
plt.axhline(y=0, color='black', linestyle='--')
plt.tick_params(axis='both', which='major', top=True, right=True, labelsize=16,
                direction='in', length=6, width=2)
ax = plt.gca()
ax.spines['top'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
plt.tight_layout()
plt.show()
# Optional export: import pandas as pd; pd.DataFrame(results).to_clipboard(index=False)